# **Adatbetöltés és Exploráció**
### 🏦 Banki Hitelkockázati Elemző Rendszer (Credit Risk Assessment)
**Készítette:** Fügedi Janka  
**Projekt célja:** Valós banki hiteligénylési adatok feldolgozása, pénzügyi törlesztési mutatók vektorizált számítása (NumPy), valamint egy objektumorientált (OOP) hitelbírálati motor megvalósítása.

---

### 1. Lépés: Adatbázis betöltése és szerkezeti ellenőrzés
Ebben a lépésben beolvassuk a 32 581 hiteligénylést tartalmazó `credit_risk_dataset.csv` állományt. Ellenőrizzük a táblázat dimenzióit, valamint megjelenítjük az első néhány rekordot az alapvető ügyfél- és hitelparaméterek (jövedelem, lakhatás, hitelösszeg, kamat) áttekintésére.

In [6]:
import pandas as pd
import numpy as np

# 1. Beolvasás Pandas DataFrame-be
df = pd.read_csv('credit_risk_dataset.csv')

# 2. Ellenőrzés: első 5 sor megtekintése
print("Sikeres betöltés! Méret:", df.shape)
display(df.head())

Sikeres betöltés! Méret: (32581, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


# **Pénzügyi Számítások (NumPy)**
### 2. Lépés: Havi törlesztőrészletek vektorizált számítása (NumPy)
A hagyományos, lassú iteratív ciklusok helyett a NumPy tömbalapú műveleteit használjuk a teljes portfólió pénzügyi mutatóinak gyors kiszámítására.

* **Alkalmazott pénzügyi képlet:** Annuitásos hiteltörlesztés ($PV$: Hitelösszeg, $r$: Havi kamatláb, $n = 36$ hónap):

  $$PMT = PV \times \frac{r(1 + r)^n}{(1 + r)^n - 1}$$
  
* A hiányzó kamatlábakat a mediánnal pótoljuk, majd a kiszámított havi részletet új oszlopként (`calculated_monthly_payment`) rögzítjük az adathalmazban.

In [7]:
# NumPy tömbök kinyerése
loan_amounts = df['loan_amnt'].to_numpy()
interest_rates = df['loan_int_rate'].fillna(df['loan_int_rate'].median()).to_numpy()

# Havi kamat és 36 hónapos futamidő
monthly_r = (interest_rates / 100) / 12
n_months = 36

# Vektorizált törlesztőrészlet (Annuity formula)
monthly_payments = loan_amounts * (monthly_r * (1 + monthly_r)**n_months) / ((1 + monthly_r)**n_months - 1)

# Eredmény hozzáadása új oszlopként a táblázathoz
df['calculated_monthly_payment'] = np.round(monthly_payments, 2)

display(df[['person_income', 'loan_amnt', 'loan_int_rate', 'calculated_monthly_payment']].head())

,person_income,loan_amnt,loan_int_rate,calculated_monthly_payment
0,59000,35000,16.02,1230.84
1,9600,1000,11.14,32.81
2,9600,5500,12.87,184.97
3,65500,35000,15.23,1217.23
4,54400,35000,14.27,1200.81


# **Objektumorientált Architektúra (OOP)**
### 3. Lépés: Objektumorientált Üzleti Modell Megvalósítása (OOP)
A szoftverarchitektúra tervezése során szétválasztjuk az adatokat és az üzleti döntési logikát a Visual Paradigmban megtervezett osztálydiagram alapján:

1. **`Customer`**: Az ügyfél demográfiai és pénzügyi paramétereit (életkor, jövedelem, munkaviszony) enkapszulálja.
2. **`Loan`**: A hitelkérelem adatait (összeg, kamat, futamidő, cél) és a törlesztőrészlet-számítás logikáját tartalmazza.
3. **`CreditEvaluation`**: A hitelelbíráló motor, amely összekapcsolja az ügyfelet a hitellel. Kiszámítja az eladósodottsági rátát (**DTI - Debt-to-Income**), és üzleti szabályok alapján besorolja a kockázatot (*Low*, *Medium*, *High Risk*).

In [8]:
from dataclasses import dataclass

@dataclass
class Customer:
    customer_id: int
    age: int
    income: float
    emp_length: float
    home_ownership: str

    def get_monthly_income(self) -> float:
        return self.income / 12 if self.income > 0 else 0.0


@dataclass
class Loan:
    loan_id: int
    amount: float
    interest_rate: float
    intent: str
    term_months: int = 36

    def calculate_monthly_installment(self) -> float:
        r = (self.interest_rate / 100) / 12
        n = self.term_months
        if r == 0:
            return self.amount / n
        return self.amount * (r * (1 + r)**n) / ((1 + r)**n - 1)


class CreditEvaluation:
    def __init__(self, customer: Customer, loan: Loan):
        self.customer = customer
        self.loan = loan

    def calculate_dti(self) -> float:
        monthly_income = self.customer.get_monthly_income()
        installment = self.loan.calculate_monthly_installment()
        return installment / monthly_income if monthly_income > 0 else 1.0

    def assess_risk(self) -> str:
        dti = self.calculate_dti()
        if dti > 0.40 or self.customer.age < 21:
            return "High Risk"
        elif dti > 0.20:
            return "Medium Risk"
        return "Low Risk"


# Teszteljük le az adathalmaz legelső sorával (Index 0)
row = df.iloc[0]
cust = Customer(0, int(row['person_age']), float(row['person_income']), float(row['person_emp_length']), row['person_home_ownership'])
loan = Loan(0, float(row['loan_amnt']), float(row['loan_int_rate']), row['loan_intent'])
evaluator = CreditEvaluation(cust, loan)

print("--- OOP MODELL TESZTJE (1. ügyfél) ---")
print(f"Ügyfél éves jövedelme: {cust.income:,.0f} USD")
print(f"Igényelt hitel: {loan.amount:,.0f} USD")
print(f"Számított havi törlesztő: {loan.calculate_monthly_installment():.2f} USD")
print(f"DTI (Törlesztő / Jövedelem): {evaluator.calculate_dti()*100:.2f}%")
print(f"Kockázati besorolás: {evaluator.assess_risk()}")

--- OOP MODELL TESZTJE (1. ügyfél) ---
Ügyfél éves jövedelme: 59,000 USD
Igényelt hitel: 35,000 USD
Számított havi törlesztő: 1230.84 USD
DTI (Törlesztő / Jövedelem): 25.03%
Kockázati besorolás: Medium Risk


# **Portfólió Szintű Összesítés**
### 4. Lépés: Portfólió szintű kockázatösszesítés (`PortfolioRiskManager`)
A banki menedzsment szintű döntéshozatalhoz nemcsak egyedi bírálatokra van szükség, hanem a teljes állomány kockázati kitettségének mérésére is.

A `PortfolioRiskManager` osztály felelőssége:
* A beérkező egyedi hitelkiértékelések gyűjtése és aggregálása.
* A teljes kintlévőség (hitelvolumen) összegzése.
* A kockázati kategóriák (*Low / Medium / High Risk*) darabszámának és eloszlásának kimutatása.

In [9]:
class PortfolioRiskManager:
    def __init__(self):
        self.evaluations = []

    def add_evaluation(self, evaluation: CreditEvaluation):
        self.evaluations.append(evaluation)

    def get_summary(self):
        total_loan_amount = sum(e.loan.amount for e in self.evaluations)
        risk_counts = {"Low Risk": 0, "Medium Risk": 0, "High Risk": 0}

        for e in self.evaluations:
            risk = e.assess_risk()
            risk_counts[risk] += 1

        return {
            "Total Applications": len(self.evaluations),
            "Total Portfolio Volume": total_loan_amount,
            "Risk Distribution": risk_counts
        }

# Futassuk le az első 1000 kérelemre demonstrációként
portfolio = PortfolioRiskManager()

for idx, row in df.head(1000).iterrows():
    # Kezeljük az esetleges hiányzó értékeket
    emp_len = float(row['person_emp_length']) if not np.isnan(row['person_emp_length']) else 0.0
    int_rate = float(row['loan_int_rate']) if not np.isnan(row['loan_int_rate']) else 11.0

    cust = Customer(idx, int(row['person_age']), float(row['person_income']), emp_len, str(row['person_home_ownership']))
    loan = Loan(idx, float(row['loan_amnt']), int_rate, str(row['loan_intent']))

    evaluation = CreditEvaluation(cust, loan)
    portfolio.add_evaluation(evaluation)

summary = portfolio.get_summary()

print("--- PORTFÓLIÓ KOCKÁZATI ÖSSZESÍTŐ (1000 kérelem) ---")
print(f"Feldolgozott kérelmek: {summary['Total Applications']} db")
print(f"Teljes hitelállomány: {summary['Total Portfolio Volume']:,.0f} USD")
print(f"Kockázati megoszlás: {summary['Risk Distribution']}")

--- PORTFÓLIÓ KOCKÁZATI ÖSSZESÍTŐ (1000 kérelem) ---
Feldolgozott kérelmek: 1000 db
Teljes hitelállomány: 17,243,775 USD
Kockázati megoszlás: {'Low Risk': 937, 'Medium Risk': 63, 'High Risk': 0}
